# Game of Life

### Description

[Game of Life](http://en.wikipedia.org/wiki/Conway's_Game_of_Life) (GoF) is a cellular automaton devised by the British mathematician John Horton Conway in 1970. The game is a zero-player game, meaning that its evolution is determined by its initial state, requiring no further input. One interacts with the Game of Life by creating an initial configuration and observing how it evolves, or, for advanced players, by creating patterns with particular properties.


The universe of the Game of Life is an infinite two-dimensional orthogonal grid of square cells, each of which is in one of two possible states, live or dead. Every cell interacts with its eight neighbours, which are the cells that are directly horizontally, vertically, or diagonally adjacent. At each step in time, the following transitions occur:

* Any live cell with fewer than two live neighbours dies, as if by needs caused by underpopulation.
* Any live cell with more than three live neighbours dies, as if by overcrowding.
* Any live cell with two or three live neighbours lives, unchanged, to the next generation.
* Any dead cell with exactly three live neighbours becomes a live cell.

The initial pattern constitutes the 'seed' of the system. The first generation is created by applying the above rules simultaneously to every cell in the seed – births and deaths happen simultaneously, and the discrete moment at which this happens is sometimes called a tick. (In other words, each generation is a pure function of the one before.) The rules continue to be applied repeatedly to create further generations.


### Assignments

* Start off implementing the GoF's rules and play with simple seeds in small dimensions
* Increase the size of the GoF's world and play with more advanced pattern
* Implement examples of the three categories of patterns *still lifes*, *oscillators* and *spaceships*
* Analyse the evolutions of the patters in terms of frequency, replication, occupancy, etc.
* ...

### Contacts 

* Marco Zanetti <marco.zanetti@unipd.it>
* Samir Suweis <samir.suweis@unipd.it>

In [1]:
import pygame
import numpy as np

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
#imposto i parametri
width, heigth = 600, 600
cell_size = 10

row = heigth // cell_size
cols = width // cell_size

FPS = 10


In [3]:
#imposto le regole

def rule(grid):
    new_grid = grid.copy()
    for x in range(1, row-1):
        for y in range(1, cols-1):
            neighbors = np.sum(grid[x-1:x+2, y-1:y+2]) - grid[x, y] #calcolo quanti vicini vivi ci sono
            
            #applico le regole
            if grid[x, y] == 1:
                if neighbors < 2 or neighbors > 3:
                    new_grid[x, y] = 0
            else:
                if neighbors == 3:
                    new_grid[x, y] = 1
    return new_grid

In [4]:
#disegno la griglia
def draw(screen, grid):
    for x in range(row):
        for y in range(cols):
            color = (0, 0, 0) if grid[x, y] == 1 else (255, 255, 255) #definisco i colori delle celle, tuple (R,G,B) in python
            rect = pygame.Rect(y*cell_size, x*cell_size, cell_size, cell_size) #definisco il rettangolo
            pygame.draw.rect(screen, color, rect) #disegna il la cella
            pygame.draw.rect(screen, (200, 200, 200), rect, 1) #disegna la griglia intorno alla cella


In [9]:
#main

pygame.init() #inizializzo i moduli di pygame
screen = pygame.display.set_mode((width, heigth)) #inizializza lo schermo
pygame.display.set_caption("Game of Life")
clock = pygame.time.Clock() #inizializza il tempo di refreshing credo na roba simile

grid = np.zeros((row, cols)) #preparo la griglia vuota

# Seed centrale
seed = np.array([[0,0,1,1],
                [0,1,1,0],
                [0,1,0,0],
                [1,1,0,0]])
x0, y0 = row//2, cols//2
grid[x0:x0+seed.shape[0], y0:y0+seed.shape[1]] = seed

running = True
paused = True  # inizia in pausa per poter disegnare

while running:
    clock.tick(FPS) #limito gli FPS, aggiornamenti al secondo, a noi non ne servono tanti

    for event in pygame.event.get(): #raccoglie gli eventi (click del mouse, della barra, etc..)
        if event.type == pygame.QUIT:# clicco sulla x per uscire
            running = False
            
    # Toggle pausa con barra spaziatrice
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE:
                paused = not paused

        # Click singolo
        if event.type == pygame.MOUSEBUTTONDOWN:
            mx, my = pygame.mouse.get_pos() #devo convertire le cordinate del cursore da pixel a coord della griglia, corrispondenza pixel-cella
            ro = my // cell_size
            col = mx // cell_size
            grid[ro, col] = 1 - grid[ro, col] #toggle matematico trasforma la cella dsa 0 a 1 o viceversa

    # Drag col mouse
    if pygame.mouse.get_pressed()[0]:  # tasto sinistro premuto
        mx, my = pygame.mouse.get_pos()
        ro = my // cell_size
        col = mx // cell_size
        grid[ro, col] = 1
    
    #fase di reddering
    # Aggiorna griglia se non in pausa
    if not paused:
        grid = rule(grid)

    screen.fill((255, 255, 255)) #colora tutto lo schermo di bianco per aggiornare altrimenti possono rimanere cose dell'immagine precedente
    draw(screen, grid) #chiama la funzione draw() per disegnare la griglia
    pygame.display.flip() #applica la griglia al tick successivo
    
pygame.quit() #chiude in modo pulito